# Thu nhập 2010–2014 — so sánh ba cách lọc đuôi trên

So sánh **bỏ top 0,1%**, **bỏ top 1%** và **bỏ trên $q_{75}+1,5\times IQR$**, tính riêng từng năm. Đây là phân tích bổ sung; mỗi bộ lọc xác định phần hộ khác nhau. Không chọn ngưỡng dựa trên p-value.

Giữ CPI 2010 = 100, CPI 2014 = 143,69; đơn vị **triệu đồng/người/năm, giá 2010**. Giữ nguyên nhãn `Quart` trong Excel. Hai năm là hai mẫu độc lập, không theo cùng hộ. Các hộ có trọng số bằng nhau.

Bốn câu hỏi: mean toàn mẫu tăng? mean từng Q tăng? phương sai giảm? gap Q4–Q1 thay đổi ra sao, xét cả chênh lệch tiền và tỷ số? **Câu phương sai hiện chỉ mô tả, chưa có quyết định kiểm định.**


In [1]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import Markdown, display

YEARS = [2010, 2014]
SCOPES = ["All", "Q1", "Q2", "Q3", "Q4"]
RULES = ["Top 0,1%", "Top 1%", "1,5×IQR"]
ALPHA = 0.05
project_directory = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "dataset/VHLSS_2014.xls").exists()
)
data_file = project_directory / "dataset/VHLSS_2014.xls"
output_directory = project_directory / "bao_cao_vhlss"
original_excel_hash = hashlib.sha256(data_file.read_bytes()).hexdigest()
cpi_by_year = pd.read_csv(output_directory / "cpi_2010_2014.csv").set_index("nam")["cpi_binh_quan_2010_bang_100"]

def format_p(value):
    return "<1e-300" if value == 0 else f"{value:.3e}"


## 1. Dữ liệu và bộ lọc

$Y=(income/size)/1000\times100/CPI$. `size` gồm mọi thành viên của hộ.

- Top p%: xếp giảm dần theo Y rồi bỏ $\lceil pn\rceil$ hộ mỗi năm. Kiểm tra đồng hạng ở ranh giới.
- IQR: $IQR=q_{75}-q_{25}$; bỏ **chỉ đuôi trên** $Y>q_{75}+1,5IQR$. Hai phân vị tính trên toàn mẫu hợp lệ của từng năm, dùng nội suy vị trí $(n+1)p$ (`method="weibull"`), giống notebook khảo sát trước.

Phân vị $q_{25},q_{75}$ dùng tính ngưỡng, khác với nhãn nhóm `Quart` gốc; không chia lại Q sau lọc. Ngưỡng áp dụng trên toàn năm, không cắt riêng trong từng Q.


In [2]:
original_by_year = {}
retained_samples = {}
retained_by_rule_year = {}
filter_rows = []

for year in YEARS:
    raw = pd.read_excel(data_file, sheet_name=str(year))
    valid = (
        np.isfinite(raw["income"]) & raw["income"].ge(0)
        & np.isfinite(raw["size"]) & raw["size"].gt(0)
        & raw["Quart"].isin([1, 2, 3, 4])
    )
    households = raw.loc[valid, ["income", "size", "Quart"]].copy()
    households["real_income_per_person"] = (
        households["income"] / households["size"] / 1000 * 100 / cpi_by_year.loc[year]
    )
    original_by_year[year] = households
    q25, q75 = np.quantile(households["real_income_per_person"], [.25, .75], method="weibull")
    iqr_cutoff = q75 + 1.5 * (q75 - q25)
    ordered = households.sort_values("real_income_per_person", ascending=False)

    for rule in RULES:
        if rule == "1,5×IQR":
            retained = households.loc[households["real_income_per_person"].le(iqr_cutoff)].copy()
            cutoff = iqr_cutoff
        else:
            fraction = 0.001 if rule == "Top 0,1%" else 0.01
            remove_count = int(np.ceil(fraction * len(households)))
            retained = ordered.iloc[remove_count:].copy()
            cutoff = retained["real_income_per_person"].max()
            assert ordered.iloc[remove_count - 1]["real_income_per_person"] > cutoff
        removed = households.loc[~households.index.isin(retained.index)]
        retained_by_rule_year[(rule, year)] = retained
        filter_rows.append({
            "rule": rule, "year": year, "original_n": len(households),
            "removed_n": len(removed), "retained_n": len(retained),
            "removed_percent": 100 * len(removed) / len(households),
            "cutoff": cutoff, "q25": q25, "q75": q75,
            "Q4_retained_n": int(retained["Quart"].eq(4).sum()),
            "all_removed_are_Q4": bool(removed["Quart"].eq(4).all()),
        })
        for scope in SCOPES:
            group = retained if scope == "All" else retained.loc[retained["Quart"].eq(int(scope[1:]))]
            retained_samples[(rule, year, scope)] = group["real_income_per_person"].to_numpy()

filter_audit = pd.DataFrame(filter_rows)
display(filter_audit)


,rule,year,original_n,removed_n,retained_n,removed_percent,cutoff,q25,q75,Q4_retained_n,all_removed_are_Q4
0,"Top 0,1%",2010,9399,10,9389,0.106394,236.250000,7.784429,22.018000,2340,True
1,Top 1%,2010,9399,94,9305,1.000106,83.296667,7.784429,22.018000,2256,True
2,"1,5×IQR",2010,9399,519,8880,5.521864,43.368357,7.784429,22.018000,1831,True
3,"Top 0,1%",2014,9398,10,9388,0.106406,175.725520,9.640531,25.945137,2339,True
4,Top 1%,2014,9398,94,9304,1.000213,89.502401,9.640531,25.945137,2255,True
5,"1,5×IQR",2014,9398,474,8924,5.043626,50.402045,9.640531,25.945137,1875,True


## 2. Estimators và phân phối — Lecture 4, Chap 3, Chap 9

Với mỗi mẫu sau lọc, tính $\bar Y$, $S^2$ (mẫu số n−1), $\widehat{Var}(\bar Y)=S^2/n$. Đây là công thức lớp, coi bộ lọc và nhãn Q như đã cho; chưa tính riêng bất định ngưỡng.

Chap 9: $JB=n[skewness^2/6+(kurtosis-3)^2/24]\approx\chi^2_2$ dưới giả thuyết dữ liệu chuẩn. Đây là kiểm tra **phân phối dữ liệu**, khác với xấp xỉ phân phối của mean. Mẫu lớn không tự làm F-test hai phương sai hợp lệ.


In [3]:
estimator_rows = []
for (rule, year, scope), values in retained_samples.items():
    sample_variance = values.var(ddof=1)
    jb = stats.jarque_bera(values)
    estimator_rows.append({
        "rule": rule, "year": year, "scope": scope, "n": len(values),
        "mean": values.mean(), "sample_variance": sample_variance,
        "mean_variance": sample_variance / len(values),
        "JB": jb.statistic,
        "JB_log10_p": -jb.statistic / (2 * np.log(10)),
        "reject_normality": bool(jb.statistic > stats.chi2.ppf(.95, 2)),
    })
estimators = pd.DataFrame(estimator_rows).set_index(["rule", "year", "scope"])
display(estimators)


n       mean  sample_variance  mean_variance  \
rule     year scope                                                    
Top 0,1% 2010 All    9389  17.598200       271.865107       0.028956   
              Q1     2350   5.256769         2.584411       0.001100   
              Q2     2349  10.382223         2.303162       0.000980   
              Q3     2350  17.092934         6.661379       0.002835   
              Q4     2340  37.743528       467.998083       0.199999   
Top 1%   2010 All    9305  16.640155       159.932951       0.017188   
              Q1     2350   5.256769         2.584411       0.001100   
              Q2     2349  10.382223         2.303162       0.000980   
              Q3     2350  17.092934         6.661379       0.002835   
              Q4     2256  34.542107       151.174406       0.067010   
1,5×IQR  2010 All    8880  14.774207        86.099336       0.009696   
              Q1     2350   5.256769         2.584411       0.001100   
              Q2     2349  10.382223         2.303162       0.000980   
              Q3     2350  17.092934         6.661379       0.002835   
              Q4     1831  29.647906        33.590529       0.018345   
Top 0,1% 2014 All    9388  20.542593       282.346593       0.030075   
              Q1     2352   6.291917         4.076711       0.001733   
              Q2     2350  13.024193         3.796978       0.001616   
              Q3     2347  20.757957         7.206633       0.003071   
              Q4     2339  42.210132       387.623724       0.165722   
Top 1%   2014 All    9304  19.670404       195.722300       0.021036   
              Q1     2352   6.291917         4.076711       0.001733   
              Q2     2350  13.024193         3.796978       0.001616   
              Q3     2347  20.757957         7.206633       0.003071   
              Q4     2255  39.418657       167.883071       0.074449   
1,5×IQR  2014 All    8924  17.824474       116.019724       0.013001   
              Q1     2352   6.291917         4.076711       0.001733   
              Q2     2350  13.024193         3.796978       0.001616   
              Q3     2347  20.757957         7.206633       0.003071   
              Q4     1875  34.635326        44.176721       0.023561   

                                JB    JB_log10_p  reject_normality  
rule     year scope                                                 
Top 0,1% 2010 All    371289.804040 -80624.556541              True  
              Q1        111.113857    -24.128067              True  
              Q2        129.618792    -28.146363              True  
              Q3        154.173940    -33.478446              True  
              Q4      48672.486108 -10569.096069              True  
Top 1%   2010 All     11158.330702  -2423.000726              True  
              Q1        111.113857    -24.128067              True  
              Q2        129.618792    -28.146363              True  
              Q3        154.173940    -33.478446              True  
              Q4       1373.585141   -298.270224              True  
1,5×IQR  2010 All      1329.687785   -288.738034              True  
              Q1        111.113857    -24.128067              True  
              Q2        129.618792    -28.146363              True  
              Q3        154.173940    -33.478446              True  
              Q4        156.155252    -33.908682              True  
Top 0,1% 2014 All     76115.169381 -16528.199026              True  
              Q1        112.545544    -24.438954              True  
              Q2        140.835847    -30.582116              True  
              Q3        133.383428    -28.963843              True  
              Q4      11902.406340  -2584.574698              True  
Top 1%   2014 All      7785.235271  -1690.542359              True  
              Q1        112.545544    -24.438954              True  
              Q2        140.835847    -30.582116              T

## 3. Mean toàn mẫu và từng Q có tăng không? — Chap 7 §7.2.4

Trong mỗi phạm vi: $\Delta=\mu_{14}-\mu_{10}$; **H₀: Δ=0; H₁: Δ>0**. Dùng **t hai mẫu độc lập, phương sai tổng thể chưa biết và không giả sử bằng nhau, một phía phải**, cùng cách bản top 0,1%.

$SE=\sqrt{S_{14}^2/n_{14}+S_{10}^2/n_{10}}$; $t=\widehat\Delta/SE$ với bậc tự do xấp xỉ theo Chap 7. Code tính từng bước và đối chiếu SciPy. CI báo hai phía 95%. Mỗi p là riêng lẻ trong phân tích bổ sung này.


In [4]:
mean_test_rows = []
for rule in RULES:
    for scope in SCOPES:
        earlier = estimators.loc[(rule, 2010, scope)]
        later = estimators.loc[(rule, 2014, scope)]
        change = later["mean"] - earlier["mean"]
        earlier_part = earlier["mean_variance"]
        later_part = later["mean_variance"]
        standard_error = np.sqrt(earlier_part + later_part)
        statistic = change / standard_error
        degrees_of_freedom = (earlier_part + later_part)**2 / (
            earlier_part**2 / (earlier["n"] - 1) + later_part**2 / (later["n"] - 1)
        )
        p_value = stats.t.sf(statistic, degrees_of_freedom)
        ci_margin = stats.t.ppf(.975, degrees_of_freedom) * standard_error
        reference = stats.ttest_ind(
            retained_samples[(rule, 2014, scope)], retained_samples[(rule, 2010, scope)],
            equal_var=False, alternative="greater",
        )
        np.testing.assert_allclose([statistic, degrees_of_freedom], [reference.statistic, reference.df])
        np.testing.assert_allclose(p_value, reference.pvalue, atol=0, rtol=1e-10)
        mean_test_rows.append({
            "rule": rule, "scope": scope,
            "n_2010": int(earlier["n"]), "n_2014": int(later["n"]),
            "mean_2010": earlier["mean"], "mean_2014": later["mean"],
            "change": change, "change_percent": 100 * change / earlier["mean"],
            "SE": standard_error, "t": statistic, "df": degrees_of_freedom,
            "p_right": p_value, "reject_H0": bool(p_value < ALPHA),
            "CI95_low": change - ci_margin, "CI95_high": change + ci_margin,
        })
mean_tests = pd.DataFrame(mean_test_rows)
display(mean_tests.style.format({"p_right": format_p}))


,rule,scope,n_2010,n_2014,mean_2010,mean_2014,change,change_percent,SE,t,df,p_right,reject_H0,CI95_low,CI95_high
0,"Top 0,1%",All,9389,9388,17.598200,20.542593,2.944393,16.731218,0.242963,12.118695,18768.211173,5.617e-34,True,2.468164,3.420622
1,"Top 0,1%",Q1,2350,2352,5.256769,6.291917,1.035148,19.691712,0.053226,19.448031,4476.962319,3.043e-81,True,0.930798,1.139498
2,"Top 0,1%",Q2,2349,2350,10.382223,13.024193,2.641970,25.447056,0.050953,51.850988,4432.113760,<1e-300,True,2.542077,2.741864
3,"Top 0,1%",Q3,2350,2347,17.092934,20.757957,3.665023,21.441743,0.076845,47.693516,4687.275956,<1e-300,True,3.514370,3.815676
4,"Top 0,1%",Q4,2340,2339,37.743528,42.210132,4.466603,11.834091,0.604749,7.385882,4636.457278,8.931e-14,True,3.281008,5.652199
5,Top 1%,All,9305,9304,16.640155,19.670404,3.030249,18.210465,0.195510,15.499194,18420.076828,3.843e-54,True,2.647031,3.413467
6,Top 1%,Q1,2350,2352,5.256769,6.291917,1.035148,19.691712,0.053226,19.448031,4476.962319,3.043e-81,True,0.930798,1.139498
7,Top 1%,Q2,2349,2350,10.382223,13.024193,2.641970,25.447056,0.050953,51.850988,4432.113760,<1e-300,True,2.542077,2.741864
8,Top 1%,Q3,2350,2347,17.092934,20.757957,3.665023,21.441743,0.076845,47.693516,4687.275956,<1e-300,True,3.514370,3.815676
9,Top 1%,Q4,2256,2255,34.542107,39.418657,4.876550,14.117696,0.376111,12.965733,4496.459047,4.515e-38,True,4.139188,5.613912


## 4. Gap tuyệt đối và tương đối

**Gap tuyệt đối:** $G_t=\mu_{Q4,t}-\mu_{Q1,t}$; **H₀: $G_{14}-G_{10}=0$; H₁: $G_{14}-G_{10}>0$**. Tương phản bốn mean, xấp xỉ chuẩn từ Chap 3: $SE_G^2=\sum_{q\in\{1,4\},t\in\{10,14\}}S_{q,t}^2/n_{q,t}$. $Z_G=\widehat\Delta_G/SE_G$; p phía phải. Giữ hướng tăng đã đổi ở bản trước sau khi xem dữ liệu; đây là phân tích bổ sung.

**Gap tương đối:** $R_t=\mu_{Q4,t}/\mu_{Q1,t}$; **H₀: $R_{14}-R_{10}=0$; H₁: $R_{14}-R_{10}\ne0$**. CLT và **xấp xỉ delta cho tỷ số — ngoài slide**, giống bản trước. Với $v_{q,t}=S_{q,t}^2/n_{q,t}$:

\[
\widehat{Var}(\widehat R_t)\approx
\frac{v_{Q4,t}}{\bar Y_{Q1,t}^2}
+\frac{\bar Y_{Q4,t}^2v_{Q1,t}}{\bar Y_{Q1,t}^4}.
\]

Cộng phương sai tỷ số của hai năm để tính $SE_R$; $Z_R=\widehat\Delta_R/SE_R\approx N(0,1)$; p hai phía. Mean Q1 dương và ước lượng phải đủ ổn định. Đây là bốn mẫu hộ, không phải hai gap/tỷ số làm hai quan sát. Không suy gap đổi từ so sánh p của các Q riêng.


In [5]:
comparison_rows = []
for rule in RULES:
    gaps, ratios, ratio_variances = {}, {}, {}
    gap_change_variance = 0.0
    for year in YEARS:
        q1 = estimators.loc[(rule, year, "Q1")]
        q4 = estimators.loc[(rule, year, "Q4")]
        assert q1["mean"] > 0
        gaps[year] = q4["mean"] - q1["mean"]
        ratios[year] = q4["mean"] / q1["mean"]
        gap_change_variance += q1["mean_variance"] + q4["mean_variance"]
        ratio_variances[year] = (
            q4["mean_variance"] / q1["mean"]**2
            + q4["mean"]**2 * q1["mean_variance"] / q1["mean"]**4
        )
    gap_change = gaps[2014] - gaps[2010]
    gap_se = np.sqrt(gap_change_variance)
    gap_z = gap_change / gap_se
    ratio_change = ratios[2014] - ratios[2010]
    ratio_se = np.sqrt(sum(ratio_variances.values()))
    ratio_z = ratio_change / ratio_se
    overall = mean_tests.loc[mean_tests["rule"].eq(rule) & mean_tests["scope"].eq("All")].iloc[0]
    variance_2010 = estimators.loc[(rule, 2010, "All"), "sample_variance"]
    variance_2014 = estimators.loc[(rule, 2014, "All"), "sample_variance"]
    comparison_rows.append({
        "rule": rule,
        "mean_2010": overall["mean_2010"], "mean_2014": overall["mean_2014"],
        "mean_change_percent": overall["change_percent"], "mean_p_right": overall["p_right"],
        "variance_2010": variance_2010, "variance_2014": variance_2014,
        "variance_change_percent": 100 * (variance_2014 / variance_2010 - 1),
        "variance_test_status": "Chưa kiểm định; chưa quyết định H0",
        "gap_2010": gaps[2010], "gap_2014": gaps[2014], "gap_change": gap_change,
        "gap_SE": gap_se, "gap_Z": gap_z, "gap_p_right": stats.norm.sf(gap_z),
        "gap_CI95_low": gap_change - stats.norm.ppf(.975) * gap_se,
        "gap_CI95_high": gap_change + stats.norm.ppf(.975) * gap_se,
        "ratio_2010": ratios[2010], "ratio_2014": ratios[2014],
        "ratio_change": ratio_change, "ratio_change_percent": 100 * ratio_change / ratios[2010],
        "ratio_SE": ratio_se, "ratio_Z": ratio_z,
        "ratio_p_two_sided": 2 * stats.norm.sf(abs(ratio_z)),
        "ratio_CI95_low": ratio_change - stats.norm.ppf(.975) * ratio_se,
        "ratio_CI95_high": ratio_change + stats.norm.ppf(.975) * ratio_se,
    })
comparison = pd.DataFrame(comparison_rows)
display(comparison[[
    "rule", "gap_2010", "gap_2014", "gap_change", "gap_p_right",
    "ratio_2010", "ratio_2014", "ratio_change", "ratio_p_two_sided",
]].style.format({"gap_p_right": format_p, "ratio_p_two_sided": format_p}))


,rule,gap_2010,gap_2014,gap_change,gap_p_right,ratio_2010,ratio_2014,ratio_change,ratio_p_two_sided
0,"Top 0,1%",32.486759,35.918215,3.431456,7.914e-09,7.179986,6.708628,-0.471358,1.490e-04
1,Top 1%,29.285338,33.126740,3.841402,2.426e-24,6.570977,6.264968,-0.306009,5.057e-04
2,"1,5×IQR",24.391137,28.343409,3.952272,3.256e-78,5.639948,5.504734,-0.135215,2.935e-02


## 5. Phương sai — hiện CHƯA kiểm định, chưa quyết định H₀

Câu hỏi: phương sai thu nhập có giảm không? **H₀: $\sigma_{14}^2=\sigma_{10}^2$; H₁: $\sigma_{14}^2<\sigma_{10}^2$**.

F-test hai phương sai ở Chap 7 §7.3 cần mô hình chuẩn. Kiểm tra chuẩn của dữ liệu không phải kiểm định giả thuyết phương sai trên. Nếu dữ liệu bác bỏ chuẩn, không dùng phân phối F này để tính p. Bảng dưới chỉ cho biết **phương sai mẫu** đổi bao nhiêu; không gọi là “không bác bỏ H₀”.


In [6]:
display(comparison[[
    "rule", "variance_2010", "variance_2014", "variance_change_percent", "variance_test_status",
]])
all_normality = estimators.reset_index().query("scope == 'All'")
display(all_normality[["rule", "year", "JB", "reject_normality"]])
display(Markdown(
    "**Trạng thái câu phương sai: chưa có kiểm định suy luận hợp lệ trong notebook này.** "
    "Không có p cho phương sai và chưa quyết định bác bỏ/không bác bỏ H₀."
))


,rule,variance_2010,variance_2014,variance_change_percent,variance_test_status
0,"Top 0,1%",271.865107,282.346593,3.855400,Chưa kiểm định; chưa quyết định H0
1,Top 1%,159.932951,195.722300,22.377721,Chưa kiểm định; chưa quyết định H0
2,"1,5×IQR",86.099336,116.019724,34.751009,Chưa kiểm định; chưa quyết định H0


,rule,year,JB,reject_normality
0,"Top 0,1%",2010,371289.804040,True
5,Top 1%,2010,11158.330702,True
10,"1,5×IQR",2010,1329.687785,True
15,"Top 0,1%",2014,76115.169381,True
20,Top 1%,2014,7785.235271,True
25,"1,5×IQR",2014,1047.173323,True


**Trạng thái câu phương sai: chưa có kiểm định suy luận hợp lệ trong notebook này.** Không có p cho phương sai và chưa quyết định bác bỏ/không bác bỏ H₀.

## 6. Kết quả so sánh

Đơn vị mean/gap: triệu đồng/người/năm, giá 2010. Phương sai dùng đơn vị bình phương. Tỷ số Q4/Q1 không có đơn vị.


In [7]:
for rule in RULES:
    result = comparison.set_index("rule").loc[rule]
    selected_means = mean_tests.loc[mean_tests["rule"].eq(rule)]
    all_q_increase = selected_means.loc[selected_means["scope"].ne("All"), "reject_H0"].all()
    absolute_decision = "bác bỏ H₀ phía tăng" if result["gap_p_right"] < ALPHA else "không bác bỏ H₀ phía tăng"
    relative_decision = "bác bỏ H₀ hai phía" if result["ratio_p_two_sided"] < ALPHA else "không bác bỏ H₀ hai phía"
    display(Markdown(
        f"**{rule}:** mean toàn mẫu {result['mean_2010']:.3f} → {result['mean_2014']:.3f} "
        f"(**{result['mean_change_percent']:+.2f}%**, p = {format_p(result['mean_p_right'])}). "
        f"Cả bốn Q có bằng chứng mean tăng: **{'có' if all_q_increase else 'không đồng thời'}**. "
        f"Gap tuyệt đối {result['gap_2010']:.3f} → {result['gap_2014']:.3f}; "
        f"{absolute_decision}, p = {format_p(result['gap_p_right'])}. "
        f"Tỷ số Q4/Q1 {result['ratio_2010']:.3f} → {result['ratio_2014']:.3f}; "
        f"{relative_decision}, p = {format_p(result['ratio_p_two_sided'])}. "
        f"Phương sai **mẫu** {result['variance_change_percent']:+.2f}%: **chỉ mô tả, chưa kiểm định**."
    ))


**Top 0,1%:** mean toàn mẫu 17.598 → 20.543 (**+16.73%**, p = 5.617e-34). Cả bốn Q có bằng chứng mean tăng: **có**. Gap tuyệt đối 32.487 → 35.918; bác bỏ H₀ phía tăng, p = 7.914e-09. Tỷ số Q4/Q1 7.180 → 6.709; bác bỏ H₀ hai phía, p = 1.490e-04. Phương sai **mẫu** +3.86%: **chỉ mô tả, chưa kiểm định**.

**Top 1%:** mean toàn mẫu 16.640 → 19.670 (**+18.21%**, p = 3.843e-54). Cả bốn Q có bằng chứng mean tăng: **có**. Gap tuyệt đối 29.285 → 33.127; bác bỏ H₀ phía tăng, p = 2.426e-24. Tỷ số Q4/Q1 6.571 → 6.265; bác bỏ H₀ hai phía, p = 5.057e-04. Phương sai **mẫu** +22.38%: **chỉ mô tả, chưa kiểm định**.

**1,5×IQR:** mean toàn mẫu 14.774 → 17.824 (**+20.65%**, p = 2.054e-90). Cả bốn Q có bằng chứng mean tăng: **có**. Gap tuyệt đối 24.391 → 28.343; bác bỏ H₀ phía tăng, p = 3.256e-78. Tỷ số Q4/Q1 5.640 → 5.505; bác bỏ H₀ hai phía, p = 2.935e-02. Phương sai **mẫu** +34.75%: **chỉ mô tả, chưa kiểm định**.

## 7. Kiểm tra và lưu kết quả

Đối chiếu t/df/p với SciPy, bản top 0,1% với các CSV chính, số hộ và nhãn Q trước/sau lọc. Không sửa dữ liệu Excel hoặc notebook chính. Chưa có trọng số/tầng/cụm khảo sát; các p/CI theo mô hình độc lập và coi ngưỡng/nhãn Q như đã cho.


In [8]:
for rule in RULES:
    for year in YEARS:
        original = original_by_year[year]
        retained = retained_by_rule_year[(rule, year)]
        audit = filter_audit.loc[filter_audit["rule"].eq(rule) & filter_audit["year"].eq(year)].iloc[0]
        assert len(retained) + audit["removed_n"] == len(original)
        assert audit["all_removed_are_Q4"]
        for q in [1, 2, 3]:
            assert set(original.loc[original["Quart"].eq(q)].index) == set(retained.loc[retained["Quart"].eq(q)].index)
        if rule == "Top 1%":
            assert audit["removed_n"] == 94

baseline_means = pd.read_csv(output_directory / "thu_nhap_bo_top_0_1_mean_tests.csv").set_index("scope")
current_means = mean_tests.loc[mean_tests["rule"].eq("Top 0,1%")].set_index("scope")
for column in ["mean_2010", "mean_2014", "change", "SE", "df", "CI95_low", "CI95_high"]:
    np.testing.assert_allclose(current_means.loc[SCOPES, column], baseline_means.loc[SCOPES, column])
current_gap = comparison.set_index("rule").loc["Top 0,1%"]
baseline_gap = pd.read_csv(output_directory / "thu_nhap_bo_top_0_1_gap_test.csv").iloc[0]
baseline_ratio = pd.read_csv(output_directory / "thu_nhap_bo_top_0_1_gap_tuong_doi.csv").iloc[0]
np.testing.assert_allclose(current_gap["gap_change"], baseline_gap["gap_change"])
np.testing.assert_allclose(current_gap["gap_p_right"], baseline_gap["p_one_sided_right"])
np.testing.assert_allclose(current_gap["ratio_change"], baseline_ratio["ratio_change"])
np.testing.assert_allclose(current_gap["ratio_p_two_sided"], baseline_ratio["p_two_sided"])
assert hashlib.sha256(data_file.read_bytes()).hexdigest() == original_excel_hash

filter_audit.to_csv(output_directory / "so_sanh_loc_thu_nhap_so_mau.csv", index=False)
estimators.reset_index().to_csv(output_directory / "so_sanh_loc_thu_nhap_phan_phoi.csv", index=False)
mean_tests.to_csv(output_directory / "so_sanh_loc_thu_nhap_mean_tests.csv", index=False)
comparison.to_csv(output_directory / "so_sanh_loc_thu_nhap_ket_qua.csv", index=False)
print("Đã đối chiếu các kết quả và lưu 4 CSV. Excel và notebook chính giữ nguyên.")


Đã đối chiếu các kết quả và lưu 4 CSV. Excel và notebook chính giữ nguyên.
